# Notebook 02 — Runner v2: Chấm bài tự động có sandbox

**Nhóm 67 | Tuần 3**

Chạy bài nộp Python qua test-case runner với 4 lớp bảo vệ:
kiểm tra syntax tĩnh (ast), phát hiện import cấm,
giới hạn thời gian 5 giây (subprocess timeout)
và giới hạn bộ nhớ 128MB (resource module).
Phân loại kết quả thành 4 loại: **SE / WA / RE / TLE**.

**Cập nhật so với tuần 2:**
- Thêm memory limit 128MB (`resource.setrlimit`)
- Chạy trong thư mục tạm riêng (`tempfile.mkdtemp`)
- Danh sách import cấm có giải thích lý do từng thư viện
- Hàm `compute_fpr()` tính False Positive Rate theo đơn vị bài nộp

# B1 - Kết nối Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Kết nối Drive')

Mounted at /content/drive
✓ Kết nối Drive


# B2 - Thiết lập đường dẫn BASE, sys.path và chạy simulate_v2


In [ ]:
import os
import json
import sys
from pathlib import Path

# Auto-resolve relative path to Project directory
BASE = Path("/content/drive/MyDrive/Project")

sys.path.insert(0, str(BASE / 'src'))
import simulate_v2

print("Dang chay mo phong sinh vien v2...")
simulate_v2.generate()


Dang chay mo phong sinh vien v2...
Dang doc du lieu tu /content/drive/MyDrive/Project/data/raw/submissions_50.json...
OK: Da sao chep 50 bai nop tu raw sang processed -> /content/drive/MyDrive/Project/data/processed/submissions_50.json
  AC: 5 bai nop
  CE: 1 bai nop
  HC: 2 bai nop
  RE: 5 bai nop
  WA: 37 bai nop
  Task IDs (42 tasks): 1 -> 50


# B3 - Import thư viện và chạy run_grading_v2


In [ ]:
import sys
import os
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE = Path('/content/drive/MyDrive/Project')
sys.path.insert(0, f'{BASE}/src')
os.chdir(BASE)

exec(open('src/run_grading_v2.py', encoding='utf-8').read())

  CHẠY CHẤM BÀI TỰ ĐỘNG V2
  Bộ test: 3 public + 6-10 hidden (Phân bố Chuông)

  ✓ Đã load 50 bài nộp mô phỏng
  ✓ Đã load 50 bài toán MBPP
  ✓ Số bài sẽ chấm: 50/50

  Bắt đầu chấm (sử dụng psutil giới hạn tài nguyên và thu thập lỗi)...

  Chấm xong 10/50 bài...
  Chấm xong 20/50 bài...
  Chấm xong 30/50 bài...
  Chấm xong 40/50 bài...
  Chấm xong 50/50 bài...
  ✓ Lưu CSV: /content/drive/MyDrive/Project/results/error_analysis_v2.csv
  ✓ Lưu JSON (Đầy đủ traceback): /content/drive/MyDrive/Project/results/error_analysis_v2.json

  KẾT QUẢ CHẤM BÀI V2 (Set 3: Bell Curve)
  Tổng submissions        : 50
  Public Test Leakage Rate: 12.0% (6 bài)
  Avg TPR public (3 test) : 32.0%
  Avg TPR hidden (10 test): 31.37%
  Latency public          : 0.0482s/test
  Latency hidden          : 0.049s/test

  Lỗi PUBLIC  → SE:3  WA:37  RE:62  TLE:0  MLE:0
  Lỗi HIDDEN  → SE:6  WA:94  RE:175  TLE:0  MLE:0

  Theo topic:
    list      : 21 bài, Leakage Rate=9.5%
    math      : 16 bài, Leakage Rate=25.0%
 

# B4 - Chạy comparison_3sets: so sánh 3 bộ test, xuất CSV/JSON


In [ ]:
import sys, os, shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Project")
sys.path.insert(0, str(BASE / 'src'))
os.chdir(BASE)

import comparison_3sets
comparison_3sets.main()

  HỆ THỐNG SO SÁNH 3 BỘ TEST TRÊN BÀI NỘP THỰC TẾ CỦA SINH VIÊN (RQ1)
  ✓ Đã load 50 bài nộp của sinh viên.
  ✓ Đã load bộ dữ liệu hidden v2: 50 bài.
  ✓ Đã load bộ dữ liệu 6 hidden: 50 bài.

  Bắt đầu chấm bài trên 3 tập cấu hình test cases...

  [SV001] sum_list() (Topic: list    ) | Actual: WA   | Set1 Pass: 0/3 | Set2 Pass: 2/9 | Set3 Pass: 2/9 
  [SV002] sum_list() (Topic: list    ) | Actual: WA   | Set1 Pass: 0/3 | Set2 Pass: 0/9 | Set3 Pass: 0/9 
  [SV003] sum_list() (Topic: list    ) | Actual: HC   | Set1 Pass: 3/3 | Set2 Pass: 6/9 | Set3 Pass: 6/9 [FP Set1]
  [SV004] is_even() (Topic: math    ) | Actual: CE   | Set1 Pass: 0/3 | Set2 Pass: 0/9 | Set3 Pass: 0/9 
  [SV005] is_even() (Topic: math    ) | Actual: WA   | Set1 Pass: 2/3 | Set2 Pass: 5/9 | Set3 Pass: 5/9 
  [SV006] reverse_string() (Topic: string  ) | Actual: WA   | Set1 Pass: 0/3 | Set2 Pass: 2/9 | Set3 Pass: 2/9 
  [SV007] find_max() (Topic: list    ) | Actual: WA   | Set1 Pass: 0/3 | Set2 Pass: 3/9 | Set3 Pass: 3/9 

# B5 - Backup kết quả lên Google Drive

In [ ]:
DRIVE_OUT = f'/content/drive/MyDrive/Project/results'
os.makedirs(DRIVE_OUT, exist_ok=True)

RESULTS_DIR = os.path.join(BASE, 'results')

for f in ['comparison_3sets.csv', 'comparison_3sets.json']:
    src = os.path.join(RESULTS_DIR, f)
    dst = os.path.join(DRIVE_OUT, f)
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
    print(f'✓ Lưu: {f}')

print('✓ Đã backup lên Drive')

✓ Lưu: comparison_3sets.csv
✓ Lưu: comparison_3sets.json
✓ Đã backup lên Drive
